[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhiksark/pycon-italy-2026-workshop/blob/main/notebooks/02-vector-add-triton.ipynb)

# 02 · Vector add in Triton - _your first Triton kernel_

**Goal:** write three lines of Triton and add two tensors on the GPU.

The kernel cell has three blank lines to fill in. Complete them. The correctness check at the bottom is your traffic light.

## Environment bootstrap

In [1]:
import importlib, subprocess, sys
def ensure(pip_name, import_name=None):
    try: importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
ensure('triton')

import torch, triton
import triton.language as tl
assert torch.cuda.is_available(), 'No GPU detected. Switch the Colab runtime to T4 GPU.'
print(f'triton {triton.__version__} | gpu {torch.cuda.get_device_name(0)}')

triton 3.1.0 | gpu NVIDIA RTX A6000


## Inputs and reference output

In [2]:
n = 1 << 20
a = torch.rand(n, device='cuda', dtype=torch.float32)
b = torch.rand(n, device='cuda', dtype=torch.float32)
reference = a + b  # torch's answer; we will compare to it

## The kernel

Three blanks. Each one is one line of Triton. Read the comments above each blank carefully - they tell you exactly what to write.

Helpful Triton API reminders:
- `tl.program_id(axis=0)` returns the integer id of this kernel instance along axis 0.
- `tl.arange(0, BLOCK_SIZE)` is a vector `[0, 1, 2, ..., BLOCK_SIZE-1]`.
- `tl.load(ptr + offsets, mask=mask)` loads a vector with masked-off elements set to 0 by default.
- `tl.store(ptr + offsets, value, mask=mask)` writes back the masked elements.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/mask-load-1d.png)

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/tile-1d.png)

Mental picture: four programs partition the input. The last program's tail is hidden by the mask.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb02-vector-add.png)

In [3]:
@triton.jit
def vec_add_kernel(
    a_ptr, b_ptr, out_ptr,
    n,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(axis=0)

    # TODO (1/3): compute `offsets`, the BLOCK_SIZE-long vector of indices this program handles.
    # Hint: `pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)`
    # offsets = ...

    # TODO (2/3): build a boolean `mask` that is True for offsets < n, False otherwise.
    # This protects the last block when n is not a multiple of BLOCK_SIZE.
    # mask = ...

    # TODO (3/3): load a and b, add them, store the result. All three steps use `mask=mask`.
    # tl.store(out_ptr + offsets, ..., mask=mask)

    pass  # remove this line once your blanks are filled in

## Launch

In [ ]:
out = torch.empty_like(a)
BLOCK_SIZE = 1024
grid = (triton.cdiv(n, BLOCK_SIZE),)
vec_add_kernel[grid](a, b, out, n, BLOCK_SIZE=BLOCK_SIZE)

## Correctness

In [ ]:
torch.testing.assert_close(out, reference)
print('correctness: ok')

## Quick timing vs `torch.add`

In [ ]:
import sys, pathlib
# Make `from notebooks.shared.*` resolve whether Jupyter was launched
# from the repo root, from notebooks/, or anywhere below.
for _root in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_root / 'notebooks' / 'shared' / 'benchmark_utils.py').is_file():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break
from notebooks.shared.benchmark_utils import median_seconds

def run_triton():
    vec_add_kernel[grid](a, b, out, n, BLOCK_SIZE=BLOCK_SIZE)
    torch.cuda.synchronize()

def run_torch():
    torch.add(a, b, out=out)
    torch.cuda.synchronize()

triton_ms = median_seconds(run_triton, runs=50, warmup=5) * 1e3
torch_ms = median_seconds(run_torch, runs=50, warmup=5) * 1e3
ratio = triton_ms / torch_ms
print(f'triton: {triton_ms:.3f} ms | torch: {torch_ms:.3f} ms | triton/torch = {ratio:.2f} (lower is better)')

# --- effective bandwidth: read a, read b, write out ---
n_bytes = 3 * a.numel() * a.element_size()
gbs = n_bytes / (triton_ms * 1e-3) / 1e9
T4_PEAK_HBM_GBS = 320.0       # T4 datasheet
print(f'effective bandwidth: {gbs:.1f} GB/s ({gbs / T4_PEAK_HBM_GBS * 100:.0f}% of T4 peak HBM)')


![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/compile-step-4.png)

## Under the hood: `@triton.jit` and the launcher

You just launched a kernel and got the right answer, but Python did not run the kernel - it **launched** it. Here is what `@triton.jit` plus `kernel[grid](...)` actually does:

1. `@triton.jit` captures the function's **source / AST** at import time. It does not execute it.
2. The first time you call `kernel[grid](a, b, out, n, BLOCK_SIZE=1024)`, Triton lowers it through:
   - **Triton-IR** - the tile-level IR, close to what you wrote.
   - **TritonGPU-IR** - adds GPU specifics: tensor layouts, register vs. shared memory.
   - **LLVM IR** - scalar.
   - **PTX** - NVIDIA's virtual ISA.
   - The CUDA driver JIT-compiles PTX to **SASS** (the real machine code for *your* GPU) and loads the resulting `cubin`.
3. The result is **cached** in `~/.triton/cache/` (override with `TRITON_CACHE_DIR`). The cache key includes every argument's dtype, every `tl.constexpr` value (e.g. `BLOCK_SIZE`), pointer alignment hints (multiples of 16 by default), and any integer args specialized as `equal_to_1`. Change any of those - fresh compile.
4. `kernel[grid](...)` is the **launcher**. It computes the cache key, fetches or compiles the artifact, packs the args into the calling convention, and calls `cuLaunchKernel` via the driver. It returns the `CompiledKernel` so you can inspect every IR stage.

Two side notes:

- **Autotune.** `@triton.autotune` compiles **one specialization per config** and benchmarks them on the first call to pick the fastest. Each config caches independently. You will see this in nb05.
- **Dumping IR for offline reading.** Run your script with `TRITON_KERNEL_DUMP=1 MLIR_ENABLE_DUMP=1 LLVM_IR_ENABLE_DUMP=1` and every stage hits the filesystem.

Let's actually look at what compiled, and prove the cache is real.

In [ ]:
# kernel[grid](...) returns the CompiledKernel for this launch.
# Re-launch (cache hit) to grab the handle without recompiling.
compiled = vec_add_kernel[grid](a, b, out, n, BLOCK_SIZE=BLOCK_SIZE)

print('IR stages held on this object:', list(compiled.asm.keys()))
print()
print('--- TTIR (Triton IR, tile-level - close to what you wrote) ---')
print(compiled.asm['ttir'])
print('--- PTX (first 30 lines - NVIDIA virtual ISA) ---')
print('\n'.join(compiled.asm['ptx'].splitlines()[:30]))

In [ ]:
# Triton caches one compiled artifact per (dtype, constexpr, alignment, ...) key.
# JITFunction.device_caches is keyed by device id; each value is a 5-tuple whose [0] holds the specialization dict.
def n_specializations():
    return sum(len(v[0]) for v in vec_add_kernel.device_caches.values())

before = n_specializations()

# New BLOCK_SIZE -> new specialization -> fresh compile.
out_alt = torch.empty_like(a)
BLOCK_ALT = 512
grid_alt = (triton.cdiv(n, BLOCK_ALT),)
vec_add_kernel[grid_alt](a, b, out_alt, n, BLOCK_SIZE=BLOCK_ALT)
mid = n_specializations()

# Same key again -> cache hit, no growth.
vec_add_kernel[grid_alt](a, b, out_alt, n, BLOCK_SIZE=BLOCK_ALT)
after = n_specializations()

print(f'specializations cached: start={before}  after_new_BLOCK_SIZE={mid}  after_repeat={after}')
print('correctness with BLOCK_SIZE=512:', torch.allclose(out_alt, a + b))

## What just happened

You wrote a GPU kernel without writing CUDA C. The grid is one-dimensional. Each program handles one BLOCK_SIZE-long chunk. The mask handles the tail when `n` is not a multiple of `BLOCK_SIZE`.

Next up: reductions - the canonical “sum a big tensor” kernel that introduces shared-memory thinking.

In [ ]:
# Bonus — when the roofline lies.
#
# The roofline says a streaming kernel like ours should sit on HBM peak.
# That's true at the limit. But at small N the kernel finishes faster than
# HBM has time to deliver peak — the bottleneck is launch overhead, not
# bandwidth. Watch where the kernel crosses over.
from notebooks.shared.benchmark_utils import median_seconds

sizes = [(1 << e) for e in (10, 14, 18, 20, 22, 24)]
print(f'{"N":>12}  {"buf":>7}  {"time":>10}  {"GB/s":>8}  {"%T4 peak":>9}')
print('-' * 56)
for n_b in sizes:
    a_b   = torch.randn(n_b, device='cuda')
    b_b   = torch.randn(n_b, device='cuda')
    out_b = torch.empty_like(a_b)
    grid_b = (triton.cdiv(n_b, 1024),)

    def run():
        vec_add_kernel[grid_b](a_b, b_b, out_b, n_b, BLOCK_SIZE=1024)
        torch.cuda.synchronize()

    runs = 500 if n_b < (1 << 16) else 100 if n_b < (1 << 22) else 30
    secs = median_seconds(run, runs=runs, warmup=20)
    bytes_moved = 3 * n_b * a_b.element_size()        # read a, read b, write out
    gbs = bytes_moved / secs / 1e9
    pct = gbs / T4_PEAK_HBM_GBS * 100
    unit, val = ('us', secs * 1e6) if secs < 1e-3 else ('ms', secs * 1e3)
    print(f'{n_b:>12,}  {n_b*4/1e6:>5.2f}MB  {val:>7.1f} {unit}  {gbs:>7.1f}  {pct:>7.0f}%')

print(
    "\nReading the table:"
    "\n  Tiny N — bandwidth looks ~0%. The kernel finishes before HBM has time"
    "\n    to deliver peak; the bottleneck is launch overhead, not bandwidth."
    "\n    The roofline has no axis for launch latency, so in this regime it"
    "\n    lies — it reports 'memory-bound at a low ceiling' instead of"
    "\n    'parallelism-bound, model does not apply.'"
    "\n  Large N — bandwidth converges to a real, measurable HBM ceiling."
    "\n    THIS is the regime where the roofline tells the truth."
    "\n  '% of T4 peak >100%' just means this card's HBM is faster than a T4's."
)
